**Proje Konusu: YSA’ların sınırları**

25*25’lik binary bir matris girişine karşılık aşağıdaki çıktıları üreten 5 YSA eğitiniz.

- A) Matriste 5 nokta (rasgele konumlarda) bulunmaktadır. Model bu noktalardan birbirine en yakın olan ikisi arasındaki Manhattan mesafesini çıktı olarak verecektir. 
- B) A’nin aynısı sadece en yakın yerine en uzak olan ikisi… 
- C) Matriste değişken sayıda (her girişte farklı sayıda olabilir) N adet (1 10 arası) nokta (rasgele konumlarda) bulunmaktadır. Model nokta sayısını çıktı olarak verecektir. 
- D) C’nin aynısı ama çıkış, nokta sayısının tek mi çift mi olduğu 
- E) C’nin aynısı ama çıkış, nokta sayısı tek ise sol üst köşeye en yakın olan noktanın sol üst köşeye Manhattan mesafesi, çift ise sağ alt köşeye en yakın olan noktanın sağ alt köşeye Manhattan mesafesi

YSA modeli olarak RNN, MLP, CNN, Transformer vb. istediğinizi kullanabilirsiniz.<br> 
Her problemde farklı birini de aynılarını da kullanabilirsiniz.

- 5 problemin her biri için: En az 800 örneğe sahip eğitim, 200 örneğe sahip test kümesi oluşturunuz.
- Her eğitim ve test kümesinde çıkışların dağılımı eşit olmalıdır. Örneğin A’da aradaki mesafeler 1-50 aralığında değişiyorsa her aralıktan eşit sayıda örnek olmalıdır. Kümeler buna göre oluşturulmalıdır.
- YSA ların hiperparametre optimizasyonunu nasıl yaptığınızı raporlayınız.
- Eğitim kümesinin çeyrek, yarım ve tam halleri için eğitim yapıp test kümesi üzerindeki performansını raporlayınız.
- Yanlış ve doğru yapılan test örneklerini kendi gözünüzle inceleyip, modellerin neyi öğrenebildiğini / öğrenemediğini yorumlayınız.
- Bulgularınızı tablolar ve grafiklerle sunun, yorumlayın.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader

import math
from collections import Counter
from util import load_dataset, plot_mat

In [2]:
GRID_SIZE = 25
device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.mps.is_available() else "cpu")

x_train, y_train, x_test, y_test = load_dataset('E')

[+] Problem E veri seti başarıyla yüklendi.
    Train -> X: (1200, 25, 25), y: (1200,)
    Test  -> X: (400, 25, 25), y: (400,)


In [3]:
class Sine(nn.Module):
    def forward(self, x):
        return torch.sin(x)

class SelfAttention(nn.Module):
    def __init__(self, embedDim):
        super(SelfAttention, self).__init__()
        self.embedDim = embedDim
        self.query = nn.Linear(embedDim, embedDim)
        self.key = nn.Linear(embedDim, embedDim)
        self.value = nn.Linear(embedDim, embedDim)

    def forward(self, x):
        Q = self.query(x)
        K = self.key(x)
        V = self.value(x)

        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.embedDim)
        attnWeights = F.softmax(scores, dim=-1)
        out = torch.matmul(attnWeights, V)
        
        return out

class ProblemECNN(nn.Module):
    def __init__(self, embedDim=32, hiddenDim=64):
        super().__init__()
        
        self.pixelEmb = nn.Linear(3, embedDim)
        
        self.parityEncoder = nn.Sequential(
            nn.Linear(1, hiddenDim),
            Sine(),
            nn.Linear(hiddenDim, hiddenDim),
            Sine(),
            nn.Linear(hiddenDim, embedDim)
        )
        
        self.attention = SelfAttention(embedDim)
        
        self.distanceHead = nn.Sequential(
            nn.Linear(embedDim, hiddenDim),
            nn.ReLU(),
            nn.Linear(hiddenDim, 1)
        )
        
        yCoords, xCoords = torch.meshgrid(torch.arange(25), torch.arange(25), indexing='ij')
        coords = torch.stack([xCoords.flatten(), yCoords.flatten()], dim=-1).float()
        self.register_buffer('coordsGrid', coords)

    def forward(self, x):
        batchSize = x.size(0)
        
        xFlat = x.view(batchSize, 625, 1)
        batchCoords = self.coordsGrid.unsqueeze(0).expand(batchSize, -1, -1)
        xWithCoords = torch.cat([xFlat, batchCoords], dim=-1)
        
        countFeature = torch.sum(xFlat, dim=1) 
        parityVector = self.parityEncoder(countFeature).unsqueeze(1)
        
        pixelFeatures = self.pixelEmb(xWithCoords)
        
        combinedFeatures = pixelFeatures + parityVector
        
        combinedFeatures = combinedFeatures * xFlat
        
        attnOut = self.attention(combinedFeatures)
        
        attnOutMasked = attnOut.masked_fill(xFlat == 0, float('-inf'))
        pooledMax = torch.max(attnOutMasked, dim=1)[0]
        
        finalOut = self.distanceHead(pooledMax).squeeze(-1)
        
        if self.training:
            return finalOut
        else:
            return torch.round(finalOut)

In [4]:
torch.manual_seed(11337)

x_train_tensor = torch.tensor(x_train).float()
y_train_tensor = torch.tensor(y_train).float()
x_test_tensor = torch.tensor(x_test).float()
y_test_tensor = torch.tensor(y_test).float()

# 2. Artık Tensör oldukları için TensorDataset sorunsuz çalışacaktır
train_dataset = TensorDataset(x_train_tensor, y_train_tensor)
test_dataset = TensorDataset(x_test_tensor, y_test_tensor)

train_loader = DataLoader(train_dataset, batch_size=512, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=512, shuffle=True)

# 2. Model, Loss ve Optimizatör
model = ProblemECNN().to(device)
model.train()

criterion = nn.MSELoss() 
optimizer = optim.AdamW(model.parameters())

epochs = 1000

In [5]:
TOLERANCE = 1.0 # Modelin tahmininin gerçek değere ne kadar uzak olabileceği (±1 piksel)

for epoch in range(epochs):
    model.train()
    trainCorrect = 0
    trainClose = 0
    trainErrorSum = 0.0
    totalSamples = 0
    
    for batch_x, batch_y in train_loader:
        batch_x = batch_x.to(device)
        batch_y = batch_y.float().squeeze().to(device)

        optimizer.zero_grad()
        predictions = model(batch_x).squeeze()

        loss = criterion(predictions, batch_y)
        loss.backward()
        optimizer.step()

        with torch.no_grad():
            predictedLabels = torch.round(predictions)
            
            # Gerçek değer ile tahmin arasındaki mutlak farkı hesapla
            absErrors = torch.abs(predictedLabels - batch_y)
            
            # 1. Katı Doğruluk (Birebir Eşleşme)
            trainCorrect += (predictedLabels == batch_y).sum().item()
            
            # 2. Yakınlık Doğruluğu (Tolerans Sınırı İçinde Kalanlar)
            trainClose += (absErrors <= TOLERANCE).sum().item()
            
            # 3. Ortalama Hata İçin Toplam Fark
            trainErrorSum += absErrors.sum().item()
            
            totalSamples += batch_y.size(0)

    trainAccuracy = trainCorrect / totalSamples
    trainCloseAcc = trainClose / totalSamples
    trainMae = trainErrorSum / totalSamples

    model.eval()
    testCorrect = 0
    testClose = 0
    testErrorSum = 0.0
    testTotal = 0
    
    with torch.no_grad():
        for batch_x, batch_y in test_loader:
            batch_x = batch_x.to(device)
            batch_y = batch_y.float().squeeze().to(device)
            
            predictions = model(batch_x).squeeze()
            predictedLabels = torch.round(predictions)
            
            absErrors = torch.abs(predictedLabels - batch_y)
            
            testCorrect += (predictedLabels == batch_y).sum().item()
            testClose += (absErrors <= TOLERANCE).sum().item()
            testErrorSum += absErrors.sum().item()
            
            testTotal += batch_y.size(0)            
            
    testAccuracy = testCorrect / testTotal
    testCloseAcc = testClose / testTotal
    testMae = testErrorSum / testTotal
    
    if (epoch + 1) % 100 == 0:
        print(f"Epoch [{epoch+1}/{epochs}] | Train Acc(±{int(TOLERANCE)}): %{trainCloseAcc*100:.1f} | Test Acc(±{int(TOLERANCE)}): %{testCloseAcc*100:.1f} | Test MAE: {testMae:.2f} px")

Epoch [100/1000] | Train Acc(±1): %22.3 | Test Acc(±1): %18.2 | Test MAE: 5.02 px
Epoch [200/1000] | Train Acc(±1): %35.1 | Test Acc(±1): %31.8 | Test MAE: 3.23 px
Epoch [300/1000] | Train Acc(±1): %81.8 | Test Acc(±1): %75.8 | Test MAE: 1.00 px
Epoch [400/1000] | Train Acc(±1): %98.6 | Test Acc(±1): %98.5 | Test MAE: 0.43 px
Epoch [500/1000] | Train Acc(±1): %100.0 | Test Acc(±1): %100.0 | Test MAE: 0.15 px
Epoch [600/1000] | Train Acc(±1): %100.0 | Test Acc(±1): %100.0 | Test MAE: 0.04 px
Epoch [700/1000] | Train Acc(±1): %100.0 | Test Acc(±1): %100.0 | Test MAE: 0.01 px
Epoch [800/1000] | Train Acc(±1): %100.0 | Test Acc(±1): %100.0 | Test MAE: 0.02 px
Epoch [900/1000] | Train Acc(±1): %100.0 | Test Acc(±1): %100.0 | Test MAE: 0.00 px
Epoch [1000/1000] | Train Acc(±1): %100.0 | Test Acc(±1): %100.0 | Test MAE: 0.00 px


In [6]:
def predict(model, x_data):
    model.eval()
    
    if not isinstance(x_data, torch.Tensor):
        x_data = torch.tensor(x_data, dtype=torch.float32)

    is_single = (x_data.dim() == 2)
    
    if is_single:
        x_data = x_data.unsqueeze(0)
        
    device = next(model.parameters()).device
    x_data = x_data.to(device)

    with torch.no_grad():
        sonuc = model(x_data)
        
    sonuc_tensor = torch.round(sonuc).int()
    
    if is_single:
        return sonuc_tensor.item()
    else:
        return sonuc_tensor.flatten()

In [7]:
wrong_indices = []
error_diffs = []

with torch.no_grad():
    if not isinstance(x_test, torch.Tensor):
        x_test = torch.tensor(x_test, dtype=torch.float32)
    x_test = x_test.to(device)
    preds = predict(model, x_test)

for i, (pred, y) in enumerate(zip(preds, y_test, strict=True)):
    if pred != y:
        # Hatanın büyüklüğünü (mutlak farkını) hesapla
        diff = abs(int(pred) - int(y)) 
        
        wrong_indices.append((i, y, pred))
        error_diffs.append(diff)

Y_EKSENI_MAKSIMUM = 100 

diff_map = dict(Counter(error_diffs))
print("\nHata Farkı Haritası (Fark: Adet):", diff_map)

if error_diffs:
    sorted_diffs = sorted(diff_map.keys())
    counts = [diff_map[d] for d in sorted_diffs]
    x_labels = [str(d) for d in sorted_diffs]

    fig_width = max(8, len(sorted_diffs) * 0.7)
    fig, ax = plt.subplots(figsize=(fig_width, 6))
    
    bars = ax.bar(x_labels, counts, color='#FF6F61', edgecolor='#CC3333', width=0.4, zorder=3)

    if len(sorted_diffs) < 6:
        ax.set_xlim(-0.5, 5.5)

    max_count = max(counts) if counts else 1
    
    # Her zaman 0'dan başlar. Tavan değeri ise belirlediğiniz limit (100) ile grafikteki en yüksek bar arasında seçilir.
    ax.set_ylim(0, max(Y_EKSENI_MAKSIMUM, max_count + (max_count * 0.1)))

    for bar in bars:
        yval = bar.get_height()
        if yval > 0: 
            # Yazıları yazdırırken üst kısımdaki boşluk (tavan yüksekliğine göre ayarlandı)
            ax.text(bar.get_x() + bar.get_width()/2, yval + (ax.get_ylim()[1] * 0.015), 
                    int(yval), ha='center', va='bottom', fontsize=11, fontweight='bold')

    ax.set_title('Yanlış Tahminlerdeki Hata Büyüklüğü Dağılımı', fontsize=14, pad=15, fontweight='bold')
    ax.set_xlabel('Mutlak Hata Farkı (|Tahmin - Gerçek|)', fontsize=12, labelpad=10)
    ax.set_ylabel('Hata Adedi', fontsize=12, labelpad=10)
    
    ax.yaxis.set_major_locator(ticker.MaxNLocator(integer=True))
    
    ax.grid(axis='y', linestyle='--', alpha=0.6, zorder=0)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    plt.tight_layout()
    plt.show()


Hata Farkı Haritası (Fark: Adet): {}


In [8]:
# Gösterilecek maksimum hatalı örnek sayısı
gosterilecek_adet = 5

print(f"\n--- İlk {gosterilecek_adet} Hatalı Tahmin Görselleştiriliyor ---")

for count, (idx, true_y, pred_y) in enumerate(wrong_indices):
    if count >= gosterilecek_adet:
        break # Sınıra ulaşınca döngüyü bitir
        
    fark = abs(int(pred_y) - int(true_y))
    print(f"\nİndeks: {idx} | Gerçek Değer: {true_y} | Tahmin: {pred_y} | Hata Payı: {fark}")
    
    # x_test GPU'da (cuda) olduğu için matplotlib sorun çıkarabilir.
    # plot_mat fonksiyonunuzun beklentisine göre x_test'i CPU'ya alıp numpy'a çeviriyoruz.
    # Eğer plot_mat doğrudan PyTorch tensörlerini destekliyorsa sadece x_test[idx] de yazabilirsiniz.
    matris = x_test[idx].cpu().numpy() if isinstance(x_test[idx], torch.Tensor) else x_test[idx]
    
    plot_mat(matris)


--- İlk 5 Hatalı Tahmin Görselleştiriliyor ---
